# LesioScan — Notebook 03: ShuffleNet V2 Training

**Goal:** train pretrained ShuffleNet V2 in two stages using **train/val only**. The final **test set remains sealed** for Notebook 05. Dataset images stay on Google Drive; no copies are created.

## 1. GPU, Drive, dan GitHub

In [ ]:
import torch
print('PyTorch:',torch.__version__)
print('GPU available:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
else: raise RuntimeError('Aktifkan GPU: Runtime → Change runtime type → T4 GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, os
REPO=Path('/content/lesioscan-ml')
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/Liex2810/lesioscan-ml.git',str(REPO)],check=True)
else: subprocess.run(['git','pull','--ff-only','origin','main'],cwd=REPO,check=True)
os.chdir(REPO)
for name in ['scripts/train_lesioscan.py','src/lesioscan/training.py','src/lesioscan/dataset.py','src/lesioscan/preprocessing.py']:
    assert (REPO/name).is_file(),f'Missing: {name}. Push file training terlebih dahulu.'
print('✅ Source Python siap')

## 2. Konfigurasi eksperimen

In [ ]:
DRIVE=Path('/content/drive/MyDrive/LesioScan-AI')
DATASET=DRIVE/'datasets/PAD-UFES-20'
SPLIT=DRIVE/'experiments/splits/split.csv'
TRAIN_OUT=DRIVE/'experiments/training/shufflenet_v2_baseline'
assert SPLIT.is_file(),SPLIT
assert DATASET.is_dir(),DATASET
TRAIN_OUT.mkdir(parents=True,exist_ok=True)
EPOCHS_HEAD=5
EPOCHS_FINETUNE=20
BATCH_SIZE=16
SEED=42
print('Input:',SPLIT,'\nOutput:',TRAIN_OUT)

## 3. Audit data tanpa membaca seluruh gambar

In [ ]:
import pandas as pd
df=pd.read_csv(SPLIT)
print('Rows:',len(df))
display(pd.crosstab(df.diagnostic,df.split))
assert len(df)==2298
assert set(df.split)=={'train','val','test'}
patients={s:set(df.loc[df.split==s,'patient_id']) for s in ['train','val','test']}
assert patients['train'].isdisjoint(patients['val'])
assert patients['train'].isdisjoint(patients['test'])
assert patients['val'].isdisjoint(patients['test'])
print('✅ Patient-level isolation benar')

## 4. Training — Stage 1 + Stage 2

**Stage 1:** Freeze backbone, train classifier head. **Stage 2:** Unfreeze all layers and fine-tune with smaller learning rate. Weighted Cross Entropy is calculated using **training labels only**. `best_shufflenetv2.pt` is selected by validation Macro-F1, never by test performance.

In [ ]:
import subprocess,sys
command=[sys.executable,'-u','scripts/train_lesioscan.py',
         '--split',str(SPLIT),'--images',str(DATASET),'--out',str(TRAIN_OUT),
         '--epochs-head',str(EPOCHS_HEAD),'--epochs-finetune',str(EPOCHS_FINETUNE),
         '--batch-size',str(BATCH_SIZE),'--seed',str(SEED)]
print('Memulai ShuffleNet V2 training...',flush=True)
subprocess.run(command,cwd=REPO,check=True)
print('✅ Training selesai; files at:',TRAIN_OUT)

## 5. Training history dan checkpoint

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
history=pd.read_csv(TRAIN_OUT/'history.csv')
display(history.tail())
with open(TRAIN_OUT/'training_summary.json',encoding='utf-8') as f: summary=json.load(f)
print(json.dumps(summary,indent=2))
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(range(1,len(history)+1),history.train_loss,label='Train loss')
ax.plot(range(1,len(history)+1),history.val_loss,label='Val loss')
ax.set(xlabel='Epoch global',ylabel='Cross entropy loss',title='ShuffleNet V2 training loss');ax.legend();plt.show()
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(range(1,len(history)+1),history.val_macro_f1,label='Val Macro-F1')
ax.set(xlabel='Epoch global',ylabel='Macro-F1',title='Validation Macro-F1');ax.legend();plt.show()
assert (TRAIN_OUT/'best_shufflenetv2.pt').is_file()
print('✅ Checkpoint exists')

## 6. Interpretasi dan langkah berikutnya

- Laporkan validation Macro-F1, balanced accuracy, train/val loss.
- Waspadai overfitting dan MEL recall yang rentan tidak stabil.
- Jangan gunakan test split untuk menentukan epoch/model.
- Benchmark DenseNet121 dan ViT-Small memakai split dan preprocessing yang setara.
- Checkpoint ini untuk eksperimen edukatif; **bukan model diagnosis klinis**.